<a href="https://colab.research.google.com/github/burningdust/ESAA/blob/main/OB_session/ESAA_OB_0904_%EC%84%B8%EC%85%98_%EB%AA%A8%EB%8D%B8%ED%9B%88%EB%A0%A8_%EC%97%B0%EC%8A%B5%EB%AC%B8%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **모델 훈련 연습 문제**
___
- 출처 : 핸즈온 머신러닝 Ch04 연습문제 1, 5, 9, 10
- 개념 문제의 경우 텍스트 셀을 추가하여 정답을 적어주세요.

### **1. 수백만 개의 특성을 가진 훈련 세트에서는 어떤 선형 회귀 알고리즘을 사용할 수 있을까요?**
___


확률적 경사 하강법이나 미니 배치 경사 하강법을 사용할 수 있다.

### **2. 배치 경사 하강법을 사용하고 에포크마다 검증 오차를 그래프로 나타내봤습니다. 검증 오차가 일정하게 상승되고 있다면 어떤 일이 일어나고 있는 걸까요? 이 문제를 어떻게 해결할 수 있나요?**
___

학습률이 너무 높아서 알고리즘이 발산하고 있다는 뜻이기 때문에 학습률을 낮춰서 다시 훈련시켜야 한다.  
모델이 훈련 데이터에 과적합되고 있을 가능성도 있으므로 규제를 추가한 모델을 사용하여 이를 해결할 수 있다.

### **3. 릿지 회귀를 사용했을 때 훈련 오차가 검증 오차가 거의 비슷하고 둘 다 높았습니다. 이 모델에는 높은 편향이 문제인가요, 아니면 높은 분산이 문제인가요? 규제 하이퍼파라미터 $\alpha$를 증가시켜야 할까요 아니면 줄여야 할까요?**
___

훈련 오차와 검증 오차가 모두 높은 경우 과소적합을 의심할 수 있으며, 이는 높은 편향이 문제인 경우이다. 편향을 줄이기 위해서는 규제 하이퍼 파라미터 alpha를 줄여야한다.

### **4. 다음과 같이 사용해야 하는 이유는?**
___
- 평범한 선형 회귀(즉, 아무런 규제가 없는 모델) 대신 릿지 회귀
- 릿지 회귀 대신 라쏘 회귀
- 라쏘 회귀 대신 엘라스틱넷

평범한 선형 회귀는 과적합될 가능성이 높으므로 모델의 가중치에 페널티를 둔 릿지 회귀를 사용하는 것이 좋다.  
릿지 회귀의 경우 상관관계가 높은 피처들이 모두 남아있을 수 있다. 라쏘 회귀는 불필요한 회귀 계수를 급격히 감소시켜 중요한 피처만 남길 수 있다.  
라쏘 회귀에서 회귀 계수 값이 급격히 변동하는 것을 완화하기 위해 릿지 회귀와 라쏘 회귀가 결합된 엘라스틱넷 회귀를 사용하는 것이 좋다.

### **추가) 조기 종료를 사용한 배치 경사 하강법으로 iris 데이터를 활용해 소프트맥스 회귀를 구현해보세요(사이킷런은 사용하지 마세요)**


---



In [3]:
import numpy as np
from sklearn.datasets import load_iris

# 1. 데이터
iris = load_iris()
X = iris["data"][:, (2, 3)]
y = iris["target"]

# 편향(Bias)을 위한 특성(x0 = 1) 추가
X_with_bias = np.c_[np.ones([len(X), 1]), X]

# 2. 데이터 분할
np.random.seed(42)
total_size = len(X_with_bias)
test_ratio = 0.2
valid_ratio = 0.2

test_size = int(total_size * test_ratio)
valid_size = int(total_size * valid_ratio)
train_size = total_size - test_size - valid_size

rnd_indices = np.random.permutation(total_size)

X_train = X_with_bias[rnd_indices[:train_size]]
y_train = y[rnd_indices[:train_size]]
X_valid = X_with_bias[rnd_indices[train_size:-test_size]]
y_valid = y[rnd_indices[train_size:-test_size]]
X_test = X_with_bias[rnd_indices[-test_size:]]
y_test = y[rnd_indices[-test_size:]]

# 3. 타겟 데이터를 원-핫 인코딩하는 함수
def to_one_hot(y):
    n_classes = y.max() + 1
    m = len(y)
    Y_one_hot = np.zeros((m, n_classes))
    Y_one_hot[np.arange(m), y] = 1
    return Y_one_hot

Y_train_one_hot = to_one_hot(y_train)
Y_valid_one_hot = to_one_hot(y_valid)
Y_test_one_hot = to_one_hot(y_test)

# 4. 소프트맥스 함수 정의
def softmax(logits):
    exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    exp_sums = np.sum(exps, axis=1, keepdims=True)
    return exps / exp_sums

# 5. 하이퍼파라미터
eta = 0.1 # 학습률
n_iterations = 5000 # 최대 훈련 에폭
m = len(X_train)
epsilon = 1e-7 # 로그 계산 오류 방지값

# 가중치 무작위 초기화 (특성 수 x 클래스 수)
n_inputs = X_train.shape[1]
n_outputs = len(np.unique(y_train))
Theta = np.random.randn(n_inputs, n_outputs)

# 조기 종료(Early Stopping) 변수 세팅
best_loss = np.inf
patience = 50 # 한계치
early_stopping_counter = 0
best_Theta = None

# 6. 배치 경사 하강법 학습 루프
for iteration in range(n_iterations):
    # 모델 예측
    logits = X_train.dot(Theta)
    Y_proba = softmax(logits)

    # 그래디언트 계산
    error = Y_proba - Y_train_one_hot
    gradients = 1/m * X_train.T.dot(error)

    # 가중치 업데이트
    Theta = Theta - eta * gradients

    # 검증 세트 손실 계산
    logits_valid = X_valid.dot(Theta)
    Y_proba_valid = softmax(logits_valid)
    xentropy_loss_valid = -np.mean(np.sum(Y_valid_one_hot * np.log(Y_proba_valid + epsilon), axis=1))

    # 조기 종료 검사
    if xentropy_loss_valid < best_loss:
        best_loss = xentropy_loss_valid
        best_Theta = Theta.copy()
        early_stopping_counter = 0
    else:
        early_stopping_counter += 1

    if early_stopping_counter >= patience:
        print(f"조기 종료. 에포크: {iteration}, 최적 검증 손실: {best_loss:.4f}")
        break

# 최적 가중치 복원 및 테스트 세트 평가
Theta = best_Theta
logits_test = X_test.dot(Theta)
Y_proba_test = softmax(logits_test)
y_predict = np.argmax(Y_proba_test, axis=1)

accuracy = np.mean(y_predict == y_test)
print(f"테스트 세트 정확도: {accuracy * 100:.2f}%")

테스트 세트 정확도: 96.67%
